# Stima H3 area-weighted

Questo notebook dimostra `estimate_h3_census_indexes()` con un piccolo layer censuario sintetico. In questo modo puoi studiare l'API senza scaricare dati ISTAT e senza avere un `census_data.gpkg` reale.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import geopandas as gpd
from shapely.geometry import box

from istat_census_data import estimate_h3_census_indexes


## Creare un layer censuario finale minimale


In [ ]:
source = gpd.GeoDataFrame(
    {
        "SEZ2021": ["5809100001", "5809100002"],
        "PRO_COM": ["58091", "58091"],
        "P1": [100, 200],
        "P2": [40, 80],
    },
    geometry=[
        box(12.0, 41.0, 12.01, 41.01),
        box(12.01, 41.0, 12.02, 41.01),
    ],
    crs="EPSG:4326",
)

aoi = gpd.GeoDataFrame(
    geometry=[box(12.0, 41.0, 12.02, 41.01)],
    crs="EPSG:4326",
)


Il workflow finale del pacchetto scrive layer chiamati `census<anno>`, ad esempio `census2021`. Il file sintetico qui sotto usa la stessa convenzione.


In [ ]:
with TemporaryDirectory() as tmp_dir:
    census_data_path = Path(tmp_dir) / "census_data.gpkg"
    source.to_file(census_data_path, layer="census2021", driver="GPKG")

    h3_estimates = estimate_h3_census_indexes(
        census_data_path=census_data_path,
        year=2021,
        aoi=aoi,
        indexes=[],
        workers=1,
    )

h3_estimates[["h3_index", "year", "h3_resolution", "P1", "P2"]].head()


Le colonne degli indicatori additivi come `P1` e `P2` sono conteggi interi nella tabella attributi H3. Le colonne di densita, quando presenti, restano valori float.


## Validare i totali

`total_validation` salva la validazione continua pesata per area usata per controllare la conservazione geometrica. `integer_total_validation` salva il controllo post-allocazione per i conteggi interi scritti nella tabella H3.


In [ ]:
h3_estimates.attrs["total_validation"], h3_estimates.attrs["integer_total_validation"]


Per dati reali, passa il percorso di un `census_data.gpkg` finalizzato, scegli gli indicatori dal catalogo, mantieni la risoluzione H3 fissa al livello 11 e conserva il default `boundary_ring_size=3` salvo casi specifici di studio del bordo AOI.
